In [54]:
!pip install -q langchain==1.0.5 langchain-community==0.4.1 langchain-groq==1.0.0 langchain-google-genai==3.0.1 chromadb==1.3.4 pypdf python-dotenv pyngrok

In [55]:
!pip install langchain-core langchain-community langchain-google-genai langchain-text-splitters chromadb -q

In [56]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
GOOGLE_MAP_API = userdata.get('GOOGLE_MAP_API')
print("API key loaded ✅")

API key loaded ✅


In [57]:
import json

with open("/content/bengaluru_places_rag.json") as f:
  data = json.load(f)

print(f"✅ Loaded {len(data['places'])} places from JSON")

✅ Loaded 37 places from JSON


In [58]:
import json
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings


places = data["places"]

documents = []
for place in places:
    content = f"""
Name: {place['name']}
Address: {place['address']}
Rating: {place['rating']} ({place['rating_count']} reviews)
Hours: {place['hours']}
Budget: {place['budget']['range']} — {place['budget']['notes']}
Description: {place['description']}
Highlights: {', '.join(place['highlights'])}
Best for (who): {', '.join(place['tags']['who'])}
Best for (time): {', '.join(place['tags']['time'])}
Best for (vibe): {', '.join(place['tags']['vibe'])}
""".strip()

    documents.append(Document(
    page_content=content,
    metadata={
        "name": place["name"],
        "budget_level": place["budget"]["level"],
        "who": ", ".join(place["tags"]["who"]),
        "time": ", ".join(place["tags"]["time"]),
        "vibe": ", ".join(place["tags"]["vibe"]),
        "rating": place["rating"],
        "address": place["address"],
        "latitude": place["latitude"],
        "longitude": place["longitude"]
    }
))
print(f"✅ {len(documents)} places loaded as documents")

splitter = CharacterTextSplitter(chunk_size=600, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"✅ {len(chunks)} chunks created")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ ChromaDB vector store ready — Lila's knowledge base is live")

✅ 37 places loaded as documents
✅ 37 chunks created
✅ ChromaDB vector store ready — Lila's knowledge base is live


In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=5000
)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are Lila, a warm and helpful travel and hangout companion for people in Bengaluru.
Based on the places information below, suggest the most relevant options.
For each place give the name, why it fits the mood, budget, and highlights.
Keep it friendly and concise.

Places information:
{context}

User's mood and request:
{question}

Your suggestions:
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Lila's chain is ready")

✅ Lila's chain is ready


In [60]:
import requests

def get_nearby_places(lat, lon):
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"

    params = {
        "location": f"{lat},{lon}",
        "radius": 1500,
        "type": "tourist_attraction",
        "key": GOOGLE_MAP_API
    }

    response = requests.get(url, params=params)
    data = response.json()

    places = []
    for place in data.get("results", [])[:5]:
        places.append(place["name"])

    return places



In [61]:
places = get_nearby_places(12.9763, 77.5929)
print(places)

["CSI St. Mark's Cathedral", 'Visvesvaraya Industrial & Technological Museum', 'Queen Victoria Statue', 'Venkatappa Art Gallery Bengaluru', 'Government Museum']


### Add Weather and Google Maps Integration




In [62]:
import requests
from google.colab import userdata

OPENWEATHER_API_KEY = userdata.get('OPENWEATHER_API_KEY').strip()

print("OpenWeatherMap API key loaded (if available) ✅")

OpenWeatherMap API key loaded (if available) ✅


In [63]:
def get_weather(lat, lon, api_key):
    if not api_key:
        return "Weather API key not found. Cannot fetch weather."

    base_url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": api_key,
        "units": "metric" # For Celsius, use "imperial" for Fahrenheit
    }
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status() # Raise an exception for HTTP errors
        weather_data = response.json()
        if weather_data and weather_data.get('main') and weather_data.get('weather'):
            temp = weather_data['main']['temp']
            description = weather_data['weather'][0]['description']
            return f"{temp}°C, {description.capitalize()}"
        else:
            return "Weather data not available."
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            return "Weather API key is unauthorized or invalid. Please check your OPENWEATHER_API_KEY."
        else:
            return f"Error fetching weather: {e}"
    except requests.exceptions.RequestException as e:
        return f"Error fetching weather: {e}"

In [64]:
def enrich_place(place_name, lat, lon):
    weather = get_weather(lat, lon)
    nearby = get_nearby_places(lat, lon)

    return {
        "place": place_name,
        "weather": weather,
        "nearby": nearby
    }

In [65]:
import requests
from google.colab import userdata

EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')

print("Exhangerate api  key loaded (if available) ✅")

Exhangerate api  key loaded (if available) ✅


In [66]:
import re

def get_exchange_rate(from_currency, to_currency, api_key):
    if not api_key:
        return "Currency API key not found. Cannot fetch exchange rates."

    # Using ExchangeRate-API as an example (replace with your chosen API endpoint)
    base_url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/{from_currency}"
    try:
        response = requests.get(base_url)
        response.raise_for_status() # Raise an exception for HTTP errors
        data = response.json()
        if data and data['result'] == 'success' and to_currency in data['conversion_rates']:
            return data['conversion_rates'][to_currency]
        else:
            print(f"Error: Exchange rate data not available for {to_currency}.")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching exchange rate: {e}")
        return None

def convert_budget(budget_range_str, exchange_rate, to_currency='USD'):
    if not exchange_rate:
        return f"Budget: {budget_range_str} (conversion not available)"

    # Assuming budget_range_str is like '₹200–₹800 per person' or similar
    numbers = re.findall(r'(\d+(?:\.\d+)?)', budget_range_str.replace('₹', '').replace(',', ''))

    if numbers:
        converted_numbers = [float(n) * exchange_rate for n in numbers]
        # Format to 2 decimal places and prepend currency symbol
        currency_symbol = {'USD': '$', 'EUR': '€', 'GBP': '£'}.get(to_currency.upper(), f'{to_currency} ')
        if len(converted_numbers) == 2:
            return f"{currency_symbol}{converted_numbers[0]:.2f}–{currency_symbol}{converted_numbers[1]:.2f} per person"
        elif len(converted_numbers) == 1:
            return f"{currency_symbol}{converted_numbers[0]:.2f} per person"
    return f"Budget: {budget_range_str} (conversion failed)"

Enrichment


In [71]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def enrich_documents_and_pass_question(input_data):
    docs = input_data["docs"]
    question = input_data["question"]

    enriched_docs_content = []

    for doc in docs:
        latitude = doc.metadata.get("latitude")
        longitude = doc.metadata.get("longitude")

        extra_info = []

        # Weather
        if latitude and longitude:
            weather = get_weather(latitude, longitude, OPENWEATHER_API_KEY)
            extra_info.append(f"Weather: {weather}")

        # Nearby Places
        if latitude and longitude:
            nearby = get_nearby_places(latitude, longitude)
            extra_info.append(f"Nearby: {', '.join(nearby)}")

        final_content = doc.page_content + "\n\n" + "\n".join(extra_info)

        enriched_docs_content.append(final_content)

    return {
        "context": "\n\n".join(enriched_docs_content),
        "question": question
    }



# Re-define Lila's chain to include both types of enrichment
qa_chain_with_all_enrichment = (
    RunnablePassthrough.assign(
        docs = (lambda x: x["question"]) | retriever,
        question = lambda x: x["question"]
    )
    | RunnableLambda(enrich_documents_and_pass_question)
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Lila's chain with all enrichment (weather, maps, currency) is ready!")

✅ Lila's chain with all enrichment (weather, maps, currency) is ready!


In [72]:
test_queries = [
    {
        "question": "Plan a one-day relaxing trip in Bengaluru under ₹2000 and also show nearby attractions. Convert the budget to USD.",
        "target_currency": "USD"
    },
    {
        "question": "Suggest a calm and peaceful place in Bengaluru with a budget under ₹1500. Also include nearby places and convert cost to EUR.",
        "target_currency": "EUR"
    },
    {
        "question": "Which place in Bengaluru is good to visit today considering the weather and nearby attractions? Budget under ₹1000.",
        "target_currency": "INR"
    },
    {
        "question": "Suggest a place in Bengaluru where I can explore and also find good food nearby. Budget ₹1200, convert to USD.",
        "target_currency": "USD"
    },
    {
        "question": "Give me the best tourist place in Bengaluru under ₹800 with nearby attractions and show the budget in GBP.",
        "target_currency": "GBP"
    },
    {
        "question": "I just want to chill somewhere in Bangalore today, not too expensive. Show me nearby places too and convert price to USD.",
        "target_currency": "USD"
    },
    {
        "question": "Plan a peaceful one-day trip in Bengaluru under ₹2000. Include weather, nearby attractions, and convert everything to USD. Explain why it's a good choice.",
        "target_currency": "USD"
    }
]

In [73]:
for i, query in enumerate(test_queries, 1):
    print(f"\n🔹 Test Case {i}")
    print("Question:", query["question"])
    print("-" * 50)

    response = qa_chain_with_all_enrichment.invoke(query)

    print(response)
    print("=" * 80)


🔹 Test Case 1
Question: Plan a one-day relaxing trip in Bengaluru under ₹2000 and also show nearby attractions. Convert the budget to USD.
--------------------------------------------------
**Your ₹2000 (≈ $25) “relax‑and‑explore” day in Bengaluru**

| # | Place | Why it fits your mood | Budget | Highlights | Nearby spots |
|---|-------|-----------------------|--------|------------|--------------|
| 1 | **Bengaluru Fort** | Quiet, historic, no entry fee – perfect for a calm start. | ₹0 | Stone walls, ancient gateways, Kote Venkataramana Temple | KR Market, Tipu Sultan’s Summer Palace |
| 2 | **KR Market (Krishna Rajendra Market)** | Bustling yet laid‑back; great for a leisurely stroll and a bite. | ₹200–₹300 (local snacks, chai) | Fresh produce, street food stalls, colonial architecture | Bengaluru Fort, St. Mark’s Cathedral |
| 3 | **Cubbon Park** | Green oasis in the city – ideal for a relaxed walk or a quick picnic. | ₹0 (entry free) | Lush lawns, heritage trees, small lakes | Lalb